# Levantamento macro × ativos — material de discussão do Comitê de Alocação

**Para:** Alexandre Kaercher e o Comitê de Alocação da LDC Capital
**De:** Eduardo (Edu) · **Reunião:** 2026-10-14 · **Base:** 83 trimestres, 2006Q1 → 2026Q3
**Feature:** F-027 (RF-041 / CA-110, CA-111)

> ## ⚠️ A especificação **não** está congelada — nada aqui é publicável
>
> A regra que define os regimes macro (quais eixos, quantos quadrantes, qual lente é a oficial)
> está em **ADR-001 com status "Proposta"**: ninguém ratificou. O próprio pipeline sabe disso e
> grava `especificacao_congelada = false` em cada rodada — a célula seguinte **lê esse campo do
> disco** em vez de repetir a afirmação.
>
> **Consequência prática:** todo número deste caderno é **material de discussão**, não insumo de
> decisão de alocação.
>
> **E, por construção, nada aqui é previsão.** O caderno responde "em que quadrante cada trimestre
> do histórico caiu" e "em momentos parecidos, como cada classe rendeu, com quantas observações".
> Nunca uma afirmação sobre o retorno futuro de qualquer linha: não há backtest, não há otimização
> e não há recomendação de carteira.

### Como ler este caderno

Ele **não calcula nada**. Cada número vem de um arquivo já construído em `data/canonical/`, e
cada figura e cada tabela saem de um módulo versionado em `apps/`. O caderno abre arquivo, chama
função pública e mostra o resultado — é por isso que ele é conferível: o mesmo comando reconstrói
tudo (seção 10) e qualquer divergência é bug da etapa dona, não deste arquivo.

### Roteiro

| § | O que tem |
|---|---|
| **0** | o gate da especificação e a impressão digital do dataset |
| **1** | o pedido do e-mail × o que foi entregue |
| **2** | os dados: catálogo, base bruta, alinhamento trimestral, painel macro |
| **3** | regimes: linha do tempo, distribuição, os indeterminados, o teste de defasagem |
| **4** | "momentos semelhantes": os vizinhos de 2026Q3 e o que veio no trimestre seguinte |
| **5** | o mapa de calor regime × ativo |
| **6** | "em que regime estamos?", respondido três vezes |
| **7** | internacional: correlação no stress, retornos por regime, a faixa de D6 |
| **8** | limitações |
| **9** | decisões que dependem do Comitê |
| **10** | como reproduzir |

In [ ]:
# Montagem do ambiente. Nenhum cálculo: caminhos, imports e os formatadores pt-BR de `apps/`.
import json
import sys
from pathlib import Path

CANDIDATOS_RAIZ = (Path.cwd(), *Path.cwd().parents)
RAIZ = next((p for p in CANDIDATOS_RAIZ if p.joinpath("pyproject.toml").is_file()), None)
if RAIZ is None:
    MSG = (
        "raiz do repositorio nao encontrada a partir de "
        f"{Path.cwd()}. Abra este caderno de dentro do repositorio: "
        "`uv run python -m jupyter lab apps/analysis/notebooks/`"
    )
    raise FileNotFoundError(MSG)
sys.path.insert(0, str(RAIZ))

import pandas as pd
from IPython.display import SVG, Markdown, display

from apps.analysis.tabelas.ranking_regime_atual import num, pct, pp
from apps.pipeline.common.io_layers import caminho_canonical

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)

CANONICO = caminho_canonical("MANIFEST.json", raiz=RAIZ).parent
FIGURAS = caminho_canonical("figuras", raiz=RAIZ)
TABELAS = caminho_canonical("tabelas", raiz=RAIZ)

MANIFEST = json.loads(caminho_canonical("MANIFEST.json", raiz=RAIZ).read_text(encoding="utf-8"))
REL_BUILD = json.loads(
    caminho_canonical("relatorio_build.json", raiz=RAIZ).read_text(encoding="utf-8")
)
REL_V1 = json.loads(
    caminho_canonical("relatorio_regimes_v1.json", raiz=RAIZ).read_text(encoding="utf-8")
)
REL_V3 = json.loads(
    caminho_canonical("relatorio_regimes_v3.json", raiz=RAIZ).read_text(encoding="utf-8")
)
REL_INTL = json.loads(
    caminho_canonical("relatorio_internacional.json", raiz=RAIZ).read_text(encoding="utf-8")
)

print("raiz do repositorio :", RAIZ)
print("camada canonica     :", CANONICO)
print("parquets em disco   :", len(list(CANONICO.glob("*.parquet"))))
print("relatorios lidos    : build, regimes_v1, regimes_v3, internacional + MANIFEST")

---

## 0. O gate da especificação e a impressão digital do dataset

As duas perguntas que vêm antes de qualquer número: **isto é publicável?** e **de que rodada
estamos falando?** A resposta das duas está no dado, não no texto. `DATASET_SHA256` é o agregado
dos artefatos determinísticos da rodada: hash diferente significa outro dataset, e números de
rodadas diferentes nunca se comparam sem conferir.

In [ ]:
ESPEC = REL_V1["especificacao"]
AGREGADO = MANIFEST["agregado"]
GATE_V3 = REL_V3["gate_de_publicacao"]
JANELA = MANIFEST["data_do_dado"]
FORA = AGREGADO["n_artefatos_fora_do_agregado"]

display(
    Markdown(
        f"""### {ESPEC["veredito_publicacao"]}

| Campo | Valor (lido do disco) |
|---|---|
| `especificacao_congelada` | **{ESPEC["especificacao_congelada"]}** |
| `publicavel` (lente V3) | **{GATE_V3["publicavel"]}** |
| ADR | {ESPEC["adr"]} — status **{ESPEC["adr_status_em_disco"]}** |
| Decisões provisórias | {", ".join(ESPEC["decisoes_provisorias"])} |
| Versão da especificação | {ESPEC["versao_especificacao"]} |
| **`DATASET_SHA256`** | `{AGREGADO["sha256"]}` |
| Artefatos no agregado | {AGREGADO["n_artefatos"]} (fora: {FORA}) |
| Linhas somadas | {AGREGADO["linhas"]:,} |
| Janela do dado | {JANELA["primeiro_trimestre_macro"]} → {JANELA["ultimo_trimestre_macro"]} |
| `vintage` mais recente | {JANELA["vintage_max"]} |

{ESPEC["regras"]["indeterminado"]}
"""
    )
)
print("Nota do relatorio sobre o carater descritivo:")
print(REL_V1["descritivo_nao_preditivo"])

---

## 1. O pedido do e-mail × o que foi entregue

O detalhamento item por item — com a fonte efetivamente usada, o motivo de cada substituição e o
efeito na comparabilidade — está em **[`docs/product/LIMITACOES-COMITE.md`](../../../docs/product/LIMITACOES-COMITE.md)
§1 e §2**. Aqui fica só o placar, para não duplicar a página que vai à reunião.

| Item | Pedido | Status |
|---|---|---|
| **A1** | PIB (IBC-Br ou IBGE) | entregue, com ressalva de janela |
| **A2 · A4 · A7** | IPCA mensal e 12 meses · Selic Over e meta · IPCA histórico | entregues |
| **A3** | Expectativas do Focus | entregue |
| **A5** | Taxas de títulos públicos | entregue |
| **A6** | Curva de juros (ETTJ ANBIMA) | **substituída** pela curva do Tesouro Direto |
| **B1 · B2** | IMA-B / 5 / 5+ · IRF-M / 1 / 1+ | entregues |
| **B3** | CDI acumulado | entregue |
| **B4** | Setoriais e índices amplos da B3 | entregue (os 9 pedidos + UTIL) |
| **B5a** | IFIX | entregue, janela curta (2011+) |
| **B5b** | FII por segmento | **fora do pacote** |
| **C1** | Dimensionamento da alocação internacional | entregue em parte (§7) |

A leitura de uma linha "substituída" ou "fora" é sempre a mesma: nenhuma lacuna foi fechada com
estimativa, interpolação ou valor plausível. Onde a fonte não existia de graça e com 20 anos, o
item aparece declarado em vez de preenchido.

In [ ]:
PRINCIPAIS = (
    "macro_quarterly.parquet",
    "macro_quarterly_asof.parquet",
    "asset_returns_quarterly.parquet",
    "alinhado_trimestral.parquet",
    "raw_series.parquet",
    "regimes_v1.parquet",
    "regime_neighbors_v3.parquet",
    "regime_asset_metrics.parquet",
)
escala = pd.DataFrame(MANIFEST["artefatos"]).set_index("arquivo")
display(
    escala.loc[
        list(PRINCIPAIS),
        ["linhas", "colunas", "date_min", "date_max", "pct_nulos", "determinista"],
    ].style.format({"linhas": "{:,.0f}", "colunas": "{:,.0f}", "pct_nulos": "{:.3f}"})
)

---

## 2. Os dados

Quatro camadas, cada uma com um papel.

1. **Catálogo de séries** (`series_catalog`) — o inventário do que o projeto sabe buscar, com URL
   e janela observada de cada série. É onde se vê o que **não** entrou.
2. **Base bruta empilhada** (`raw_series`) — toda série na **frequência nativa**, longa, com
   `source`, `series_id`, `vintage` e `collected_at` em cada linha. É aqui que se responde "de
   onde veio este valor?".
3. **Alinhamento trimestral** (`alinhado_trimestral`) — o valor de cada série no fim de cada
   trimestre, com o número de dias úteis de carregamento (`ffill_dias_uteis`) e o motivo quando a
   célula fica vazia. Lacuna acima de 5 dias úteis **não** é preenchida.
4. **Painel macro** (`macro_quarterly`) — 83 trimestres × 42 variáveis, uma coluna por variável, a
   tabela que a classificação de regime lê.

In [ ]:
CATALOGO = REL_BUILD["series_catalog"]
BRUTO = REL_BUILD["raw_series"]

display(
    Markdown(
        f"""| Medida | Valor |
|---|---|
| Séries no catálogo | **{CATALOGO["linhas"]}** |
| Séries presentes em `data/processed/` | **{CATALOGO["presentes_no_processed"]}** |
| Séries que cobrem 1T2006 | {CATALOGO["cobrem_1t2006"]} |
| Linhas na base bruta | **{BRUTO["linhas"]:,}** |
| Janela da base bruta | {BRUTO["date_min"]} → {BRUTO["date_max"]} |
| Linhas duplicadas descartadas | {REL_BUILD["linhas_dedup"]} |
| Nulos na coluna de valor | {BRUTO["pct_nulos_value"]:.3f}% |
"""
    )
)
display(pd.DataFrame([BRUTO["linhas_por_source"]], index=["linhas por fonte"]).T)
display(pd.DataFrame([BRUTO["linhas_por_freq"]], index=["linhas por frequencia nativa"]).T)

In [ ]:
alinhado = pd.read_parquet(caminho_canonical("alinhado_trimestral.parquet", raiz=RAIZ))
macro = pd.read_parquet(caminho_canonical("macro_quarterly.parquet", raiz=RAIZ))

print("alinhado_trimestral :", alinhado.shape, "- 3 linhas de amostra")
display(
    alinhado.head(3)[
        ["quarter", "quarter_end", "series_id", "date_obs", "value", "ffill_dias_uteis", "source"]
    ]
)
print("macro_quarterly     :", macro.shape, "- 3 linhas de amostra (primeiras colunas)")
display(macro.head(3)[["quarter", "ibcbr_yoy", "ibcbr_gap", "ipca_12m", "ipca_12m_delta"]])
print("as 42 colunas do painel macro:")
print(", ".join(macro.columns))

In [ ]:
# Os três valores-canário: a mesma data e o mesmo valor que a fonte original publica.
# Nenhuma conta aqui — recorte por `series_id` e `date` sobre a base bruta.
CANARIOS = {
    "B3_IBOV": ("2006-02-01", "38.484,84"),
    "ANBIMA_IMAB": ("2003-09-16", "723,070317"),
    "MSCI_ACWI_NETR": ("2006-01-31", "121,3858"),
}
bruto = pd.read_parquet(
    caminho_canonical("raw_series.parquet", raiz=RAIZ),
    columns=["series_id", "source", "date", "value", "unit"],
)
recorte = bruto.loc[
    bruto["series_id"].isin(CANARIOS)
    & bruto["date"].isin(pd.to_datetime([d for d, _ in CANARIOS.values()]))
]
display(recorte.sort_values(["series_id", "date"]).reset_index(drop=True))
print("valor publicado pela fonte, para conferencia linha a linha:")
for serie, (data, na_fonte) in CANARIOS.items():
    print(f"  {serie:<16} {data}  fonte: {na_fonte}")

---

## 3. Regimes macro: a linha do tempo e a distribuição

A lente **V1** corta os 83 trimestres em quatro quadrantes, cruzando dois eixos: o **hiato de
atividade** (IBC-Br dessazonalizado contra a própria média móvel de 20 trimestres) e a
**aceleração da inflação** (variação da taxa de 12 meses do IPCA). Eixo nulo em T ⇒ o trimestre
sai **INDETERMINADO** com o motivo escrito, nunca com o quadrante "mais provável".

A figura abaixo sai de `apps/analysis/figuras/timeline_regimes.py` (F-025); os dados exatamente
como plotados ficam ao lado dela, em `figuras/timeline_regimes_dados.parquet`.

In [ ]:
display(SVG(filename=str(caminho_canonical("figuras/timeline_regimes_revised.svg", raiz=RAIZ))))

In [ ]:
DIST = REL_V1["distribuicao"]
INDET = REL_V1["indeterminados"]

distribuicao = pd.DataFrame(DIST["por_grupo"]).T
distribuicao.index.name = "rotulo V1"
display(distribuicao[["n", "alerta_n_baixo", "conclusao_suprimida", "alerta_desbalanceamento"]])

display(
    Markdown(
        f"""**Os {INDET["n"]} trimestres sem quadrante** ({INDET["pct_da_amostra"]:.2%} da amostra),
todos pelo mesmo motivo — `{", ".join(INDET["por_motivo"])}`:

`{"` · `".join(INDET["trimestres"])}`

{INDET["motivo_de_fundo"]}
"""
    )
)

### O teste de defasagem (`asof` × `revised`), em três linhas

1. O mesmo histórico é construído **duas vezes**: a versão **descritiva**, que usa a série como ela
   está hoje, e a versão **defasada**, em que o valor do trimestre T é o último que já tinha sido
   divulgado até o fim de T.
2. O percentual de trimestres que **trocam de quadrante** entre as duas versões é a medida de
   quanto a classificação depende de informação que não existia na época — e ele passou do limite
   de 25% escrito em CA-074.
3. A conclusão, portanto, é sobre o **uso**: a V1 responde bem "em que quadrante cada trimestre do
   histórico caiu" e **não** responde de forma confiável "em que quadrante estamos agora, com o que
   se sabe hoje". É a diferença entre descrever e operar — e é por isso que a versão defasada
   aparece sempre rotulada, nunca no lugar da outra.

In [ ]:
DIV = REL_V1["divergencia_asof_vs_revised"]
display(
    Markdown(
        f"""| Medida | Valor |
|---|---|
| Trimestres comparáveis | {DIV["trimestres_comparaveis"]} |
| Trocam de rótulo | **{DIV["trimestres_que_trocam"]}** |
| Percentual que troca | **{DIV["pct_que_trocam"]:.2%}** |
| Limiar escrito (CA-074) | {DIV["limiar_inoperante"]:.0%} |
| `modelo_operacional` | **{DIV["modelo_operacional"]}** |

{DIV["texto"]}
"""
    )
)
display(pd.DataFrame(DIV["trocas"]).head(10))

---

## 4. "Momentos semelhantes": os vizinhos de 2026Q3

Esta é a lente que responde literalmente à pergunta do e-mail — *"como cada classe de ativo e até
setores performaram em momentos semelhantes"*. O trimestre de referência (2026Q3) é descrito por um
vetor de cinco variáveis macro padronizadas e comparado com todos os outros trimestres da janela
pela distância de Mahalanobis, com a covariância encolhida (Ledoit-Wolf, intensidade **medida**).
Saem os **8** trimestres mais próximos.

Duas coisas que essa lista **não** é: não é previsão de regime, e o retorno medido é o do trimestre
**seguinte** a cada vizinho — nunca o do próprio vizinho, que seria olhar para dentro da janela que
definiu a semelhança.

E um resultado que vale citar na reunião: a lista dos 8 vizinhos é **exatamente a mesma** quando o
sistema é proibido de olhar qualquer trimestre futuro. Essa é a lente que passa no teste de
defasagem da seção 3.

In [ ]:
VIZ = REL_V3["vizinhos_do_comite"]
SHRINK = REL_V3["shrinkage"]
SENS = REL_V3["sensibilidade"]["somente_passado_vs_descritiva"]

print("trimestre de referencia :", VIZ["quarter_ref"])
print("metrica de distancia    :", VIZ["lente"]["metrica"])
print("shrinkage (rho medido)  :", num(SHRINK["intensidade_rho"]))
print("cobertura               :", REL_V3["cobertura"]["com_vetor_completo"], "de 83 trimestres")
print("features no vetor       :", REL_V3["features"]["n_features"])
print("descartada              :", ", ".join(f["nome"] for f in REL_V3["features"]["descartadas"]))

features = pd.DataFrame(REL_V3["features"]["usadas"])
display(features[["nome", "chave", "origem", "unidade", "cobertura", "substitui"]])

vizinhos = pd.DataFrame(VIZ["vizinhos"])
display(
    vizinhos[
        ["rank", "quarter_neighbor", "quarter_t_mais_1", "distancia", "status_regime_v1_neighbor"]
    ].style.format({"distancia": "{:.4f}"})
)
print(
    "a mesma lista sem olhar o futuro:",
    f"jaccard = {num(SENS['jaccard'])},",
    f"em comum = {SENS['em_comum']} de {SENS['n_a']}",
)

In [ ]:
RANK = REL_V3["ranking_ativos_t_mais_1"]
print("metrica :", RANK["metrica_do_ranking"])
print("o que e :", RANK["o_que_e_medido"])
print("limiares:", RANK["limiares"])
print("trimestres T+1 :", ", ".join(sorted(RANK["trimestres_t_mais_1"])))

ranking = pd.DataFrame(RANK["ativos"]).set_index("asset_id")
display(
    ranking[
        [
            "n",
            "media",
            "mediana",
            "erro_padrao",
            "hit_ratio_cdi",
            "alerta_n_baixo",
            "conclusao_suprimida",
        ]
    ].style.format(
        {
            "media": pct,
            "mediana": pct,
            "erro_padrao": pp,
            "hit_ratio_cdi": "{:.1%}",
        }
    )
)

A coluna `erro_padrao` está em pontos percentuais por trimestre e é a que importa mais que a ordem:
com `n = 8`, a margem de erro da média de uma linha é da mesma magnitude da diferença entre linhas
vizinhas do ranking. Em palavras: **a ordem não é estatisticamente distinguível**. O número
sustenta "este grupo rendeu mais que aquele, com esta incerteza" — e as duas travas automáticas do
pacote saem daí: `n < 8` entra com alerta e `n < 5` sai sem conclusão (coluna
`conclusao_suprimida`).

---

## 5. O mapa de calor regime × ativo

Linhas = os 24 ativos, agrupados por classe. Colunas = os quatro quadrantes da V1 **mais** a coluna
dos vizinhos de 2026Q3. Célula = excesso sobre o CDI (por razão, nunca por subtração), com o valor
**e** o `n` escritos dentro.

**Como ler, em oito linhas** (as convenções de desenho de F-023):

1. **Azul bateu o CDI, vermelho perdeu.** A escala é divergente e **simétrica** em ±extremo — sem a
   simetria, o cinza do meio deixaria de significar "empatou".
2. **Verde está fora de propósito:** verde × vermelho é o par que colapsa em protanopia e
   deuteranopia, e é exatamente a convenção que o reflexo de mercado usaria.
3. **Três estados de confiança, cada um em dois canais** — cor nunca sozinha: `n ≥ 8` com valor em
   destaque; `5 ≤ n < 8` com borda tracejada e `*`; `n < 5` perde a cor divergente.
4. **O cinza de supressão não é o neutro da escala.** Se fosse o mesmo, "sem conclusão" leria como
   "empatou com o CDI" — que é uma afirmação.
5. **A ordem das linhas é declarada, nunca por ranking:** por classe, e dentro da classe por
   duração ou porte, que é a ordem com que o Comitê fala de carteira.
6. **O `CDI` fecha a figura e sai identicamente zero.** É a conferência visual da própria métrica:
   se a linha dele não for zero, o número está errado antes da figura.
7. **A figura não recalcula nada.** Excesso, `n` e as duas flags vêm da tabela de métricas; as
   flags são reconferidas contra os limiares da lente e divergência é erro, não hachura própria.
8. **No SVG cada célula tem texto de apoio** (erro-padrão, acerto contra o CDI, mediana, pior
   trimestre). No PNG, não — se o objetivo é inspecionar, abra o `.svg`.

A segunda figura é a vista de **controle**: ela cruza os horizontes (V1 medida em T+1, vizinhos
medidos em T0) para separar o efeito do **horizonte** do efeito da **lente**. Ela não substitui a
principal.

In [ ]:
display(SVG(filename=str(caminho_canonical("figuras/heatmap_v1_revised.svg", raiz=RAIZ))))

In [ ]:
# A vista de horizonte cruzado está sendo renomeada de `v3_t1` para `horizonte_cruzado`
# (fechamento da Onda 4). O caderno resolve o nome em vez de depender de qual já está em disco.
CANDIDATOS_CRUZADO = (
    "figuras/heatmap_horizonte_cruzado.svg",
    "figuras/heatmap_v3_t1.svg",
)
cruzado = next(
    (
        caminho_canonical(nome, raiz=RAIZ)
        for nome in CANDIDATOS_CRUZADO
        if caminho_canonical(nome, raiz=RAIZ).is_file()
    ),
    None,
)
print("vista de horizonte cruzado resolvida em:", cruzado.name if cruzado else "AUSENTE")
if cruzado is None:
    print("gere com: uv run python -m apps.analysis.figuras.heatmap_regime_ativo")
else:
    display(SVG(filename=str(cruzado)))

---

## 6. "Em que regime estamos?" — a mesma pergunta, três respostas

A página abaixo sai inteira de `apps/analysis/tabelas/ranking_regime_atual.py` (F-024) e é renderizada
aqui como ela vai ao Comitê. Ela responde a pergunta **três vezes** — V1 descritiva, V1 defasada e
"momentos semelhantes" — e **não escolhe** entre elas, porque essa escolha é a decisão **D1**, que
está aberta (seção 9).

Duas coisas para olhar com atenção na leitura: a resposta da V1 descritiva para 2026Q3 é
`INDETERMINADO` e a página usa um **fallback declarado** (o último trimestre com quadrante), com a
coluna `e_fallback` marcada em todas as linhas — o fallback nunca é chamado de "o regime atual"; e
a concordância entre as três lentes é **baixa de propósito visível**, medida por Spearman sobre a
ordem inteira e por Jaccard no topo.

In [ ]:
display(
    Markdown(
        caminho_canonical("tabelas/ranking_regime_atual.md", raiz=RAIZ).read_text(encoding="utf-8")
    )
)

In [ ]:
display(
    Markdown(
        caminho_canonical("tabelas/concordancia_lentes.md", raiz=RAIZ).read_text(encoding="utf-8")
    )
)

---

## 7. Internacional: o que o dado diz sobre o tamanho da alocação

Item **C1** do e-mail — e a metade do objetivo para a qual os blocos A e B não listam nenhuma
série. Três leituras, nesta ordem.

**(a) O câmbio como amortecedor.** A correlação entre a variação do dólar e o IBOV é medida em
blocos de trimestres, sempre com o `n` de pares. O achado central é que ela é bem mais negativa nos
trimestres marcados como **stress** do que nos calmos — o amortecimento aparece justamente quando é
necessário. A regra de stress é explícita: queda da inclinação real da curva acima de um
desvio-padrão (estimado só com o passado) **ou** alta do dólar acima de 10% no trimestre. O ramo de
risco-país ficou **fora, declarado**, porque a série morreu em 2024 e preenchê-la seria inventar dado.

**(b) Retornos por regime.** O internacional em reais no mesmo arcabouço de quadrantes, com `n` em
cada célula. Nenhuma linha mistura moedas: o S&P em dólar fica fora desta tabela de propósito — é a
mesma série em outra moeda, e uma coluna de retorno que mistura moedas não tem leitura.

**(c) A faixa de D6.** O pedido de "um número" é respondido com **faixa**: piso estrutural mais
banda tática por regime. Com `n` de 13 a 19 trimestres e quatro ativos, precisão pontual nessa
amostra é falsa — e a própria faixa muda materialmente com uma restrição de apresentação.

> **A tabela de variância mínima por regime não entra neste caderno.** Ela existe em
> `internacional_pesos.parquet` com `publicavel_na_v1 = false`, porque a **Anti-SPEC §10.5** põe
> variância mínima por regime e fronteira eficiente **fora** da entrega de 2026-10-14. O que é
> publicável de F-026 são os retornos por regime, as correlações, a regra de stress e a **faixa** —
> é isso que está abaixo.

In [ ]:
STRESS = REL_INTL["regra_de_stress"]
HEDGE = REL_INTL["hedge_cambial"]["stress_vs_nao_stress"]

print("formula :", STRESS["formula"])
print("serie de inclinacao :", STRESS["serie_de_inclinacao"])
print("amostra do desvio   :", STRESS["amostra_do_desvio"])
print("ramo fora           :", STRESS["ramo_embi_fora"])
print(
    f"marcados: {STRESS['n_stress']} de {STRESS['n_trimestres']} trimestres"
    f" (inclinacao {STRESS['n_so_pelo_ramo_inclinacao']},"
    f" cambio {STRESS['n_so_pelo_ramo_usdbrl']}, ambos {STRESS['n_pelos_dois_ramos']})"
)
print(
    "trimestres de stress:",
    ", ".join(t["quarter"] for t in STRESS["trimestres_de_stress"]),
)

correlacoes = pd.DataFrame(HEDGE["blocos"]).T
correlacoes.index.name = f"bloco — par {HEDGE['par']}"
display(
    correlacoes[["correlacao", "n_pares", "n_trimestres_do_bloco"]].style.format(
        {"correlacao": "{:.3f}"}
    )
)
print(HEDGE["leitura"])

In [ ]:
intl = pd.read_parquet(caminho_canonical("internacional_por_regime.parquet", raiz=RAIZ))
print("lentes na tabela:", ", ".join(sorted(intl["lente"].unique())))
display(
    intl.loc[intl["lente"] == "V1"]
    .set_index(["regime", "asset_id"])[
        ["n_obs", "ret_medio", "excesso_cdi", "ret_real", "hit_ratio_cdi", "corr_ibov"]
    ]
    .sort_index()
    .style.format(
        {
            "ret_medio": pct,
            "excesso_cdi": pct,
            "ret_real": pct,
            "hit_ratio_cdi": "{:.1%}",
            "corr_ibov": "{:.3f}",
        }
    )
)

In [ ]:
D6 = REL_INTL["faixa_de_d6"]
print("decisao  :", D6["decisao"])
print("forma    :", D6["forma"])
print("ativo    :", D6["ativo_representativo"])
print("selo     :", D6["selo"])
print(
    "piso estrutural :",
    pct(D6["piso_estrutural"]["min"], sinal=False),
    "a",
    pct(D6["piso_estrutural"]["max"], sinal=False),
)
print(
    "banda tatica    :",
    pct(D6["banda_tatica"]["min"], sinal=False),
    "a",
    pct(D6["banda_tatica"]["max"], sinal=False),
)

# Só as colunas publicáveis na v1: regime, `n`, piso, banda e as flags.
# O peso de variância mínima fica FORA (Anti-SPEC §10.5) — ver a nota da seção.
faixa = pd.DataFrame(D6["linhas"]).set_index("regime")
faixa["piso_estrutural"] = faixa["piso_estrutural"].astype(str)
faixa["banda_tatica"] = faixa["banda_tatica"].astype(str)
display(
    faixa[
        [
            "n_obs",
            "piso_estrutural",
            "banda_tatica",
            "truncado",
            "alerta_n_baixo",
            "conclusao_suprimida",
        ]
    ].sort_index()
)
display(Markdown(f"> {D6['numero_pontual_rejeitado']}\n>\n> {D6['distincao_obrigatoria']}"))

---

## 8. Limitações

A página completa, escrita para ser lida inteira antes da reunião, é
**[`docs/product/LIMITACOES-COMITE.md`](../../../docs/product/LIMITACOES-COMITE.md)**. As dez que
mais mudam o que se pode afirmar:

1. **A especificação não está congelada.** ADR-001 em "Proposta": todo número é material de
   discussão, e o pacote carrega esse selo em cada figura e cada tabela.
2. **`n` é pequeno.** Com dispersão trimestral de 12%, a margem de erro da média de uma célula fica
   em torno de 2,8 pp por trimestre com n = 19 e 4,2 pp com n = 8. A ordem do ranking não é
   estatisticamente distinguível.
3. **A classificação troca de rótulo em um terço dos trimestres** entre a versão descritiva e a
   defasada — acima do limite de 25% de CA-074. A V1 descreve o histórico; não opera o presente.
4. **A crise de 2008 fica fora do eixo de crescimento.** Os 16 trimestres indeterminados são
   2006Q1–2009Q3 e 2026Q3, porque o IBC-Br começa em 2003 e a média móvel de 20 trimestres só fecha
   em 2009Q4. Nada foi estimado no lugar.
5. **Quebra estrutural: 2006-2008 não é o mesmo país que o pós-2016.** Retorno nominal não é
   comparável entre épocas — daí a métrica-cabeça ser excesso sobre o CDI — e restringir a janela ao
   pós-2016 troca a maioria dos "momentos semelhantes" de 2026Q3.
6. **Janelas próprias:** ICON desde 2007, IMOB desde 2008, IFIX desde 2011. A alternativa seria
   cortar toda a análise em 2011 e perder a janela de 20 anos do e-mail.
7. **Índice não é ativo investível.** Todas as linhas são índices de carteira teórica revisada:
   sem taxa, corretagem, spread, imposto ou custo de rebalanceamento. Sharpe de índice não é Sharpe
   implementável, e a diferença é material nas linhas menos líquidas.
8. **Convenções que mudam o número** estão medidas e declaradas: trimestre é o último dia útil;
   excesso sobre o CDI é razão, nunca subtração; o nível trimestral do IBC-Br é a média dos três
   meses; macro vem em por cento e retornos em fração — cruzar as duas sem olhar a unidade erra por
   cem vezes.
9. **Substituições de fonte:** a curva vem do Tesouro Direto, não da ETTJ oficial da ANBIMA (que só
   existe no produto pago); os índices IMA vêm do repositório público da ANBIMA; o bloco
   internacional vem de Yahoo e MSCI. Cada uma está registrada com motivo e efeito.
10. **O que não está no pacote:** FII por segmento, risco-país, termos de troca, ouro, S&P com
    hedge cambial, a lente V2 completa — e, por decisão de escopo, nenhum backtest, nenhuma
    otimização e nenhuma recomendação de carteira. Nada aqui é previsão de retorno nem de regime.

---

## 9. As doze decisões que dependem do Comitê

Nenhuma destas respostas veio de uma pessoa: todas são o padrão provisório da pesquisa, aplicado
para que a estrutura pudesse ser construída agora. Virar qualquer uma é legítimo e previsto — a
coluna da direita diz o preço. O detalhamento está em
**[`LIMITACOES-COMITE.md` §7](../../../docs/product/LIMITACOES-COMITE.md)**.

| # | Decisão | Padrão provisório | Preço de virar |
|---|---|---|---|
| **D1** | qual lente é a oficial | V1 oficial, "momentos semelhantes" como leitura central, V2 secundária | com a V2 oficial a tabela principal vai a 12 estados com 5 a 15 observações; com a V3 oficial o entregável central passa a ser a lista de vizinhos |
| **D3** | eixo de inflação da V1 | aceleração do IPCA | "nível vs. meta" exige a série histórica de metas do CMN, que não está na base, e **os 83 rótulos mudam** |
| **D5** | métrica principal do mapa de calor | excesso sobre o CDI | nominal torna 2006 e 2020 incomparáveis; as três colunas já existem, é trocar um parâmetro |
| **D6** | tamanho do internacional | piso de 16-20% + banda tática de 15-35% | "100% tático" exigiria otimizar com ~20 observações por regime e entregar número pontual |
| **D12** | setoriais e FII segmentado no pacote | *best effort*, com `n` e data de início visíveis | "completo" põe extratores no caminho crítico; "fora" reduz o mapa de calor |
| **L1** | IFIX só desde 2011 | janela própria, sem IFIX sintético | painel balanceado restringe **toda** a análise a 2011-2026 |
| **L2** | FII por segmento via provedor privado | autorizado só como leitura qualitativa — e **não foi extraído** | se autorizado, entra com ~2 ciclos macro; se negado, B5b fica formalmente não atendido |
| **L4** | curva oficial da ANBIMA é paga | não contratar; usar a curva do Tesouro Direto | contratar recalcula a inclinação histórica e passa a envolver credencial |
| **D-i** | qual inclinação é a variável de regime | 5 anos − 3 anos (95,25% de cobertura) | a 5a−1a tem 60,53%: ~40% dos trimestres cairiam num dia sem valor |
| **D-s** | nível trimestral do IBC-Br | média dos 3 meses | voltar ao último mês faz 7 trimestres mudarem de quadrante |
| **D-t** | movimento de 25 pb na Selic | conta como ciclo | com a regra estrita, os cortes reais de 25 pb de 2024 voltam a ser "parada" |
| **D-u** | eixo de crescimento dos "momentos semelhantes" | expectativa de PIB do Focus | sem o Focus a lente **não tem vetor para 2026Q3** — o trimestre da pergunta — e perde 2006-2009 |

**Atalho:** ratificar os padrões provisórios de uma vez muda o status dos ADRs de "Proposta" para
"Aceito" e abre o fluxo formal de entrega. Enquanto isso não acontecer, o gate da seção 0 segue
fechado.

---

## 10. Como reproduzir — e como citar uma versão

Nada neste caderno foi digitado à mão. **Ambiente:** `uv sync --extra dev`; nenhuma credencial é
necessária, todas as fontes do caminho crítico são públicas e gratuitas.

```bash
# 1. reconstruir a camada canônica a partir do bruto já arquivado, sem rede
uv run python -m apps.pipeline run --offline

# 2. conferir: recalcula o sha256 de cada artefato contra o MANIFEST.json
#    (código de saída diferente de zero se qualquer um divergir)
uv run python -m apps.pipeline verify

# 3. congelar a rodada que vai à reunião (grava MANIFEST.json + FROZEN.txt)
uv run python -m apps.pipeline freeze --tag comite-2026-10-14

# variações úteis
uv run python -m apps.pipeline run --offline --listar-etapas   # a ordem das etapas
uv run python -m apps.pipeline run --offline --so regimes_v3   # retomar uma etapa só
uv run python -m apps.pipeline run --online                    # com os extratores (precisa de rede)
```

As figuras e as tabelas saem de módulos próprios e entram na rodada pela etapa de análise:

```bash
uv run python -m apps.analysis.figuras.timeline_regimes
uv run python -m apps.analysis.figuras.heatmap_regime_ativo
uv run python -m apps.analysis.tabelas.ranking_regime_atual
```

E este caderno:

```bash
uv run python -m jupyter lab apps/analysis/notebooks/
```

**Reprodutibilidade medida:** duas execuções `--offline` sobre o mesmo bruto produzem os artefatos
determinísticos byte a byte idênticos, e o `MANIFEST.json` também. O `ldc.duckdb` fica fora do
agregado de propósito: é derivado e descartável, e seus bytes mudam a cada escrita.

**Como citar um número:** sempre com o `DATASET_SHA256` da seção 0 e com o `sha256` do artefato de
onde ele saiu — a tabela abaixo. Hash diferente significa outro dataset; números de rodadas
diferentes nunca se comparam sem conferir.

In [ ]:
hashes = pd.DataFrame(MANIFEST["artefatos"]).set_index("arquivo")
display(
    hashes[["bytes", "linhas", "determinista", "sha256"]].style.format(
        {"bytes": "{:,.0f}", "linhas": "{:,.0f}"}, na_rep="—"
    )
)
print("DATASET_SHA256 (agregado da rodada):", MANIFEST["agregado"]["sha256"])
print("convencao do agregado:", MANIFEST["convencao"]["agregado"])
print("excluidos do agregado:", ", ".join(MANIFEST["convencao"]["excluidos_do_agregado"]))
print("congelamento:", MANIFEST["congelamento"])

---

### Fechamento

Este caderno é o entregável **F-027** (RF-041 / CA-110, CA-111). Ele não contém lógica de negócio:
todo cálculo vive em `apps/pipeline/` e todo desenho em `apps/analysis/{figuras,tabelas}/`, e há
teste que falha se uma função ou uma conta aparecer aqui. Ele vai ao repositório com os outputs
limpos — o que você está vendo é o resultado de uma execução sobre o `data/canonical/` desta
máquina, e é reproduzível pelos comandos da seção 10.

**Antes de qualquer envio ao Comitê:** o gate da seção 0 precisa estar aberto (ADR-001 ratificado)
e o fluxo formal de entrega precisa estar cumprido — revisão humana registrada, dataset congelado
com versão e hash, linha no log do projeto e rollback descrito. Sem isso, o envio é bloqueado por
construção, e não por esquecimento.

*Dúvida sobre um número? Ele tem endereço: o artefato está na tabela da seção 10, a janela e o `n`
estão na própria célula, e a origem de cada linha está em `raw_series` (`source`, `series_id`,
`vintage`, `collected_at`).*